### This notebook demonstrates how to build and evaluate a neural network regression model using TensorFlow and Keras. We will follow these steps:
1. Load and preprocess the data
2. Build the neural network model
3. Train the model
4. Evaluate the model's performance
5. Perform cross-validation and hyperparameter tuning

### 1. Load and prepare the dataset ###

In [1]:
### Import libraries ###
import pandas as pd
import numpy as np
from tf_keras.src.losses import mean_absolute_percentage_error

import helper_func as hf
import os
import time
import random

from scipy.stats import kendalltau

from sklearn.metrics import mean_squared_error, r2_score


import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, TensorDataset
from torch import nn, optim
import torch.nn.functional as F
from torchinfo import summary

In [2]:
### define data path and loader ###
data_path = './processed_data/dataset_with_descriptors.csv'

# Path to save the model checkpoints and parameters
nn_checkpoint_path = './nn_output/checkpoints'
os.makedirs(nn_checkpoint_path, exist_ok=True)

model_path = './nn_output/nn_model_params'
os.makedirs(model_path, exist_ok=True)

#### 1.1. Load and Separate Dataset ###

In [8]:
def separate_features(data_path: str):
    """Read the dataset and separate features and target variable."""
    assert data_path.endswith('.csv') and os.path.exists(data_path), "Data path must be a valid CSV file."
    data = pd.read_csv(data_path, low_memory=False)

    features = [col for col in data.columns if col != 'Mole fraction']
    target = 'Mole fraction'
    data = data.dropna(subset=features)
    X = data[features].drop(columns=['Component 1', 'Component 2', 'Smiles 1', 'Smiles 2', 'mol1', 'mol2'])
    X = hf.convert_to_numeric(X)

    # Separate temperature and pressure features
    T_col = [col for col in X.columns if col.startswith('Temperature')]
    P_col = [col for col in X.columns if col.startswith('Pressure')]
    T_P = X[T_col + P_col]
    X = X.drop(columns=T_col + P_col)

    # Separate molecular descriptors
    X1_desc = [col for col in X.columns if col.startswith('mol1_desc')]
    X2_desc = [col for col in X.columns if col.startswith('mol2_desc')]
    X1 = X[X1_desc]
    X2 = X[X2_desc]

    # Separate one-hot encoded features
    Properties_code = [col for col in X.columns if col.startswith('property')]
    Property_values = 'value'
    Properties = X[[Property_values] + Properties_code]

    # Phase Embeddings
    Phase = X['phase_Liquid']

    y = data[target]
    return X1, X2, T_P, Properties, Phase, y

In [9]:
X1, X2, T_P, Properties, phase, y = separate_features(data_path)
print("X1 shape:", X1.shape)
print("X2 shape:", X2.shape)
print("T_P shape:", T_P.shape)
print("Properties shape:", Properties.shape)
print("Phase shape:", phase.shape)
print("y shape:", y.shape)

X1 shape: (104142, 200)
X2 shape: (104142, 200)
T_P shape: (104142, 2)
Properties shape: (104142, 26)
Phase shape: (104142,)
y shape: (104142,)


#### 1.2. Prepare Mixture Properties ###


### 2. Build the Neural Network Model ###

#### 2.1. Define the Neural Network Architecture ###
**Two different Neural Network Architectures:**
1. **BaselineNN**: A simple feedforward neural network with three hidden layers and ReLU activation functions, followed by a sigmoid activation in the output layer. This architecture serves as a basic model for regression tasks.
    * Input Layer: Takes the combined features of the mixture (molecular descriptors, temperature, pressure, and one-hot encoded properties).
    * Hidden Layers: Three fully connected layers with ReLU activation functions to capture non-linear relationships between the features and the target variable (mole fraction). `[256, 128, 64]` neurons in each hidden layer respectively.
    * Output Layer: A single neuron with a sigmoid activation function to predict the mole fraction, which is constrained between 0 and 1.
    * **Loss Function**: `Mean Absolute Percentage Error (MAPE)` is used as the loss function to optimize the model during training, as it provides a percentage-based error metric that is more interpretable for regression tasks involving small mole fractions.
2. **Mixture and Physical Property Integration**: A more complex architecture that integrates mixture properties, physical conditions (temperature, pressure and extra measured property), and molecular descriptors. This model is designed to capture the intricate relationships between these features for improved prediction of mole fractions.
    * As displayed in the following picture:
        * Three different input branches for the final MLP layers: one for the molecular structure descriptors (`SMILES-BERT`), one for the physical conditions (`temperature, pressure and the target physical property`), and one for the mixture properties (`RDKit encoded by a simple encoder`).
        * Mixture Fusion is defined as the following formula:
        $$\text{Mixture Fusion} = [\text{latent}_1, \text{latent}_2, |\text{latent}_1 - \text{latent}_2|, |\text{latent}_1 + \text{latent}_2|, \mathbf{\text{latent}_1} \odot \mathbf{\text{latent}_2}]$$
        * Final Prediction is defined as the following formula:
        $$\hat{z} = \alpha \cdot \text{MLP}_{\text{mixture}}(\text{Mixture Fusion}) + \beta \cdot \text{MLP}_{\text{physical}}(\text{Physical Conditions}) + \gamma \cdot \text{MLP}_{\text{structure}}(\text{Molecular Descriptors})$$
        $$\hat{y} = \text{Sigmoid}(\hat{z}) = \frac{1}{1 + e^{-\hat{z}}}$$
        where $\alpha$, $\beta$, and $\gamma$ are learnable parameters that determine the contribution of each feature type to the final prediction.

Í![img](./nn_architecture.png)

In [14]:
class EarlyStopping():
    """Early stopping utility to stop training when validation loss does not improve."""
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

In [15]:
def get_data_loader(data_path: str,
                    batch_size: int = 64,
                    test_size: float = 0.1,
                    val_size: float = 0.1,
                    random_state=42):
    """
    Load and preprocess the dataset, then create DataLoaders for training, validation, and testing.
    :param data_path:
    :param batch_size:
    :param test_size:
    :param val_size:
    :param random_state:
    :return: data loaders for training, validation, and testing
    """
    # 1. Read the data and preprocess
    assert data_path.endswith('.csv') and os.path.exists(data_path), "Data path must be a valid CSV file."
    data = pd.read_csv(data_path, low_memory=False)
    features = [col for col in data.columns if col != 'Mole fraction']
    target = 'Mole fraction'
    data = data.dropna(subset=features)
    X = data[features].drop(columns=['Component 1', 'Component 2', 'Smiles 1', 'Smiles 2', 'mol1', 'mol2'])
    X = hf.convert_to_numeric(X)
    y = data[target]
    X_np = X.to_numpy(dtype=float)
    # 2. Convert to PyTorch tensors
    X_tensor = torch.tensor(X_np, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.float32).view(-1, 1)

    # 3. Create a TensorDataset and split into train/val/test
    dataset = TensorDataset(X_tensor, y_tensor)
    total_size = len(dataset)
    test_size = int(test_size * total_size)
    val_size = int(val_size * total_size)
    train_size = total_size - test_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=torch.Generator().manual_seed(random_state))

    # 4. Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    return train_loader, val_loader, test_loader

def eval_nn(data_loader,
            model,
            device,
            print_out: bool = False) -> tuple:
    """Evaluate a neural network model on a given dataset.
     :param data_loader: DataLoader providing the evaluation dataset.
     :param model: The neural network model to evaluate.
     :param device: The device (CPU or GPU) to perform evaluation on.
     :return: MSE and R2 (prints the Mean Squared Error and R^2 Score)."""

    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            all_preds.append(outputs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    mape = mean_absolute_percentage_error(all_targets, all_preds)
    r2 = r2_score(all_targets, all_preds)

    if print_out:
        print(f"Mean Squared Error: {mape:.4f}")
        print(f"R^2 Score: {r2:.4f}")

    return (mape, r2)


def train_nn(model,
             train_loader,
             val_loader,
             args):
    """Train a neural network model using the provided training and validation data loaders.
     :param model: The neural network model to train.
     :param train_loader: DataLoader providing the training dataset.
     :param val_loader: DataLoader providing the validation dataset.
     :param args: A dictionary containing training parameters such as learning rate, number of epochs, and checkpoint path."""

    model.to(args.device)
    criterion = nn.MSELoss()
    optimizer = args.optimizer(model.parameters(), lr=args.learning_rate, weight_decay=args.weight_decay)
    early_stopping = EarlyStopping(patience=args.patience, min_delta=args.min_delta)
    best_val_loss = float('inf')

    train_losses, val_losses = [], []
    for epoch in range(args.num_epochs):
        model.train()
        running_loss = 0.0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(args.device), targets.to(args.device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = args.criterion(outputs, targets)
            optimizer.step()

            running_loss += loss

        train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validate the model
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for x, y_true in val_loader:
                x, y_true = x.to(args.device), y_true.to(args.device)
                y_hat = model(x)
                loss = args.criterion(y_hat, y_true)
                val_loss += loss

        val_loss /= len(val_loader.dataset)
        val_losses.append(val_loss)
        print(f"Epoch {epoch+1}/{args.num_epochs}, Training Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        # Save the best model checkpoint
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(args.checkpoint_path, 'best_model.pth'))

        if args.early_stop:
            early_stopping(val_loss)
            if early_stopping.early_stop:
                print("Early stopping triggered. Stopping training.")
                break

    # Save the final model parameters and results
    torch.save(model.state_dict(), os.path.join(args.checkpoint_path, 'final_model.pth'))
    np.save(os.path.join(args.checkpoint_path, 'train_losses.npy'), np.array(train_losses))
    np.save(os.path.join(args.checkpoint_path, 'val_losses.npy'), np.array(val_losses))

    if args.training_curves:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Training Loss')
        plt.plot(val_losses, label='Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training and Validation Loss Curves')
        plt.legend()
        plt.show()

class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super(AttrDict, self).__init__(*args, **kwargs)
        self.__dict__ = self

In [18]:
class BaselineNN(nn.Module):
    """A simple feedforward neural network for regression tasks.
    :param input_dim: The number of input features.
    :param hidden_dims: A list of integers specifying the number of neurons in each hidden layer.
    :param output_dim: The number of output features (default is 1 for regression).
    The network architecture consists of three fully connected layers with ReLU activation functions, followed by a sigmoid activation in the output layer."""

    def __init__(self, input_dim, hidden_dims=[256, 128, 64], output_dim=1):
        super(BaselineNN, self).__init__()
        self.name = "SimpleMLP"
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim = output_dim
        self.fc1 = nn.Linear(self.input_dim, self.hidden_dims[0])
        self.fc2 = nn.Linear(self.hidden_dims[0], self.hidden_dims[1])
        self.fc3 = nn.Linear(self.hidden_dims[1], self.hidden_dims[2])
        self.fc4 = nn.Linear(self.hidden_dims[2], self.output_dim)


    def forward(self, x):
        x = x.view(-1, self.input_dim)  # Flatten the input

        x = F.relu(self.fc1(x)) # Layer 1 with ReLU activation
        x = F.relu(self.fc2(x)) # Layer 2 with ReLU activation
        x = F.relu(self.fc3(x)) # Layer 3 with ReLU activation
        return F.sigmoid(self.fc4(x)) # Output layer with sigmoid activation

In [19]:
train_loader, val_loader, test_loader = get_data_loader(data_path, batch_size=32)
input_dim = next(iter(train_loader))[0].shape[1]
model = BaselineNN(input_dim=input_dim, output_dim=1)
print(summary(model, input_size=(1, input_dim)))

Layer (type:depth-idx)                   Output Shape              Param #
BaselineNN                               [1, 1]                    --
├─Linear: 1-1                            [1, 256]                  110,080
├─Linear: 1-2                            [1, 128]                  32,896
├─Linear: 1-3                            [1, 64]                   8,256
├─Linear: 1-4                            [1, 1]                    65
Total params: 151,297
Trainable params: 151,297
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.15
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.61
Estimated Total Size (MB): 0.61


#### 2.2. Train the Baseline Neural Network Model (MLP) ###

In [ ]:
args = AttrDict()
baseline_dict = {
    'device': torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'optimizer': optim.AdamW,
    'criterion': mean_absolute_percentage_error,
    'learning_rate': 1e-3,
    'weight_decay': 1e-5,
    'num_epochs': 100,
    'checkpoint_path': nn_checkpoint_path,
    'early_stop': True,
    'patience': 10,
    'min_delta': 0.001,
    'training_curves': True
}

args.update(baseline_dict)

# Reference List:
1. HANNA: hard-constraint neural network for consistent activity coefficient prediction [link](https://pubs.rsc.org/en/content/articlehtml/2024/sc/d4sc05115g)
2. SMILES-BERT: Large Scale Unsupervised Pre-Training for Molecular Property Prediction [link](https://www.researchgate.net/publication/335704451_SMILES-BERT_Large_Scale_Unsupervised_Pre-Training_for_Molecular_Property_Prediction)
3. Graph neural networks for temperature-dependent activity coefficient prediction of solutes in ionic liquids [link](https://www.sciencedirect.com/science/article/pii/S0098135423000224)
4. Neural network prediction model for dew point and bubble point phase equilibria behavior of binary mixtures in alcohol systems [link](
https://www.sciencedirect.com/science/article/pii/S0009250924006821)
5. Vapor-liquid phase equilibria behavior prediction of binary mixtures using machine learning [link](https://www.sciencedirect.com/science/article/pii/S0009250923009144)